# 00 -- Setup & Environment Check

This notebook confirms your environment is ready for the hands-on session:
the conda env, the installed `cassa-photometry` package, a working **plate
solver** for Phase 2, the sky data it will use, and the provided dataset.

> Edit the paths in the config cell below to match the layout you were given.

In [ ]:
# ============================================================
#  WORKSHOP CONFIG
# ============================================================
# One shared module rather than this cell copied into six notebooks, so a
# path is changed once and the notebooks cannot drift apart.
# Override any path with an environment variable; see workshop_config.py.
import importlib, os, sys

_here = os.path.dirname(os.path.abspath('workshop_config.py'))
if _here not in sys.path:
    sys.path.insert(0, _here)

# Reloaded, not merely imported. A kernel that imported workshop_config before
# the file was edited keeps serving the cached module, and the first name added
# since then fails much further down as a bare NameError -- which is exactly how
# `raw_frames()` broke for anyone whose kernel predated it.
import workshop_config
importlib.reload(workshop_config)
from workshop_config import *   # noqa: F403  (RAW_DIR, WORK_DIR, PHASE*_DIR, ...)

require_dataset()   # fails now, with the command that fixes it, not later
os.makedirs(WORK_DIR, exist_ok=True)
show_config()

## 1. The package imports and reports its version

In [ ]:
import cassa_photometry
from cassa_photometry.config import load_config
print('cassa_photometry', cassa_photometry.__version__)
cfg = load_config()
print('default FWHM =', cfg.phase3.fwhm)

## 2. Where the sky data will come from
Phase 2 plate-solves, and the pipeline works out which sky data your field
needs and fetches only that -- about 6 MB for ASTAP, ~165 MB of index files
for Astrometry.net -- caching it under `~/.cache/cassa-photometry/`.

**A full local set, if one exists, is used in preference and nothing is
downloaded.** The pipeline looks for `./astrometry_data` relative to the working
directory, which from `workshop/notebooks/` is the wrong place, so
`workshop_config` points `CASSA_ASTROMETRY_INDEX` at the set that ships beside
`workshop/`. A directory already named in your environment wins.

`0 index files` here is **not** a problem: it just means the tiles will be
fetched during the solve.

In [ ]:
from cassa_photometry.config import load_config

config = load_config()
idx = config.resolve_astrometry_index_dir()
n = len([f for f in os.listdir(idx) if f.startswith('index-')]) if os.path.isdir(idx) else 0
print('Local index set: ', idx)
print('                 ', 'exists' if os.path.isdir(idx) else 'absent', f'({n} index files)')
print('Index cache:     ', config.resolve_index_cache_dir())

from cassa_photometry.astap_db import resolve_db_dir
astap_dir = resolve_db_dir(config)
tiles = len(os.listdir(astap_dir)) if os.path.isdir(astap_dir) else 0
print('ASTAP tiles:     ', astap_dir, f'({tiles} cached)')

## 3. A plate solver is available
Three backends can solve, and **any one is enough**. `install.sh` tries ASTAP,
then Astrometry.net's `solve-field`, then an in-process Python solver, and uses
the first that installs.

Your results do not depend on which one you have: the astrometric residual
`ASTRMS` -- which Phase 3 uses to size its catalog cross-match -- is measured by
the pipeline itself rather than taken from the solver, so every backend writes
the same header. The `ASTRMSRC` card records which route was used.

What matters is that **at least one** backend is usable.

In [ ]:
from cassa_photometry.phase2_integration.solvers import BACKENDS, available_backends

usable = available_backends(config)
for name in BACKENDS:
    print(f"  {'[ok]' if name in usable else '[--]'}  {name}")

if usable:
    print(f"\nPhase 2 will use: {usable[0]}")
else:
    print("\nNO PLATE SOLVER -- Phase 2 cannot solve a WCS, so Phase 3 has no")
    print("zero point. Run `cassa-doctor` for the fix.")

## 4. The dataset is where we expect it

In [ ]:
# The raw tree is whatever the acquisition software wrote. A night arrives as
#     <date>/BIAS|DARK|FLAT|LIGHT/<target>/<frame>.fits
# and a simulated set as one flat directory; `raw_frames()` walks either, which
# is exactly how Phase 1 finds them (cassa_photometry.paths.find_raw_frames).
raw = raw_frames()
print(len(raw), 'raw frames found under', RAW_DIR)

for relative, count in raw_layout().items():
    print(f'  {relative:<40s} {count:4d}')

print()
for f in raw[:5]:
    print(' ', os.path.relpath(f, RAW_DIR))

If all four checks pass, continue to **01 -- The Data Model**.

> Note: Phase 3 (notebook 04) cross-matches against online reference catalogs
> (APASS / Pan-STARRS / SDSS), so that step needs outbound internet on the node.